In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. Load Data

In [2]:

df_sales = pd.read_csv("sales.csv")

## Feature Engineering

In [3]:

df_sales["date"] = pd.to_datetime(df_sales["date"])
df_sales = df_sales.sort_values("date").reset_index(drop=True)


df_sales["year"] = df_sales["date"].dt.year
df_sales["month"] = df_sales["date"].dt.month
df_sales["day"] = df_sales["date"].dt.day
df_sales["is_state_holiday"] = (
    (df_sales["state_holiday"].astype(str).str.strip() != "0")
).astype(int)

In [4]:
### Train only on open days with actual sales

In [5]:

df_trainable = df_sales[(df_sales["open"] == 1) & (df_sales["sales"] > 0)].copy()

X = df_trainable.drop(columns=["sales", "date", "state_holiday", "open"])
y = df_trainable["sales"]

### 2. Train / Validation / Test Sequential Split (70% / 15% / 15%)

In [6]:
# --- 
total_rows = len(df_trainable)
#calculates the row index where the first 70% of your data ends.
train_cutoff = int(total_rows * 0.70)

#calculates the row index where 85% of your total data ends.
val_cutoff = int(total_rows * 0.85)

X_train = X.iloc[:train_cutoff].copy()
X_val = X.iloc[train_cutoff:val_cutoff].copy()
X_test = X.iloc[val_cutoff:].copy()

y_train = y.iloc[:train_cutoff]
y_val = y.iloc[train_cutoff:val_cutoff]
y_test = y.iloc[val_cutoff:]

### 3. Target Encoding on store_ID (Using y_train only!)

In [7]:

store_means = y_train.groupby(X_train["store_ID"]).mean()
global_mean = y_train.mean()

# Map to all three splits
for df in [X_train, X_val, X_test]:
    df["store_avg_sales"] = df["store_ID"].map(store_means).fillna(global_mean)
    df.drop(columns=["store_ID"], inplace=True)

### 4. Scale Features

In [8]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### 5. Define Models

In [9]:

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=15, random_state=42, n_jobs=-1
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42
    ),
}

### 6. Train on 'Train' and Evaluate on 'Validation'

In [10]:

print("=== VALIDATION PHASE (Tuning Dashboard) ===")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    # Predict on Validation data
    y_val_pred = model.predict(X_val_scaled)

    # Metrics
    r2_v = r2_score(y_val, y_val_pred)
    mae_v = mean_absolute_error(y_val, y_val_pred)
    rmse_v = np.sqrt(mean_squared_error(y_val, y_val_pred))

    print(f"\n{name}:")
    print(f"  R² Score:  {r2_v:.4f}")
    print(f"  MAE:       ${mae_v:,.2f}")
    print(f"  RMSE:      ${rmse_v:,.2f}")

=== VALIDATION PHASE (Tuning Dashboard) ===

Linear Regression:
  R² Score:  0.7934
  MAE:       $1,019.23
  RMSE:      $1,454.02

Random Forest:
  R² Score:  0.9054
  MAE:       $687.58
  RMSE:      $983.78

XGBoost:
  R² Score:  0.9002
  MAE:       $723.34
  RMSE:      $1,010.42


### 7. Final Test Phase

In [11]:

# Assuming Random Forest wins validation, run final verification on Test:
champion_name = "Random Forest"
champion_model = models[champion_name]
y_test_pred = champion_model.predict(X_test_scaled)

print(f"\n=== FINAL TEST PHASE (Unseen Future Data) ===")
print(f"Champion Model: {champion_name}")
print(f"  Final R² Score: {r2_score(y_test, y_test_pred):.4f}")
print(f"  Final MAE:      ${mean_absolute_error(y_test, y_test_pred):,.2f}")


=== FINAL TEST PHASE (Unseen Future Data) ===
Champion Model: Random Forest
  Final R² Score: 0.8915
  Final MAE:      $713.27


# LOAD AND PREPROCESS INFERENCE DATA

In [12]:

df_new = pd.read_csv("REAL_DATA.csv") 

# Convert and clean dates
df_new['date'] = df_new['date'].astype(str).str.strip()
df_new["date"] = pd.to_datetime(df_new["date"], dayfirst=True, errors='coerce')
df_new = df_new.dropna(subset=['date'])
df_new = df_new.sort_values("date").reset_index(drop=True)

# Extract date features (EXACTLY same as training)
df_new["year"] = df_new["date"].dt.year
df_new["month"] = df_new["date"].dt.month
df_new["day"] = df_new["date"].dt.day

# Feature Engineering for is_state_holiday (same as training)
df_new["is_state_holiday"] = (
    (df_new["state_holiday"].astype(str).str.strip() != "0")
).astype(int)

# Create X_new with the same features as X_train BEFORE target encoding
X_new = df_new[['day_of_week', 'nb_customers_on_day', 'promotion', 'school_holiday', 'year', 'month', 'day', 'is_state_holiday', 'store_ID']].copy()
print(f"X_new columns before target encoding: {X_new.columns.tolist()}")


# Apply target encoding (same as training)
X_new["store_avg_sales"] = X_new["store_ID"].map(store_means).fillna(global_mean)
X_new.drop(columns=["store_ID"], inplace=True)




X_new columns before target encoding: ['day_of_week', 'nb_customers_on_day', 'promotion', 'school_holiday', 'year', 'month', 'day', 'is_state_holiday', 'store_ID']


In [13]:
# First, check what columns X_train has
print(f"X_train columns: {X_train.columns.tolist()}")

# Create a new DataFrame with the same columns as X_train
X_new_final = pd.DataFrame()

# Add a dummy 'Unnamed: 0' column to match X_train
X_new_final['Unnamed: 0'] = range(len(X_new))

X_train columns: ['Unnamed: 0', 'day_of_week', 'nb_customers_on_day', 'promotion', 'school_holiday', 'year', 'month', 'day', 'is_state_holiday', 'store_avg_sales']


In [14]:
# Add all the other columns from X_new
for col in X_new.columns:
    X_new_final[col] = X_new[col]

# Now X_new_final has the same columns as X_train
print(f"X_new_final columns: {X_new_final.columns.tolist()}")

X_new_final columns: ['Unnamed: 0', 'day_of_week', 'nb_customers_on_day', 'promotion', 'school_holiday', 'year', 'month', 'day', 'is_state_holiday', 'store_avg_sales']


In [15]:
# Verify columns match exactly
if list(X_new_final.columns) == list(X_train.columns):
    print("\n✓ Columns match exactly! Ready to scale.")
else:
    print("\n⚠️ Column mismatch!")
    print(f"X_new_final: {X_new_final.columns.tolist()}")
    print(f"X_train: {X_train.columns.tolist()}")


✓ Columns match exactly! Ready to scale.


In [16]:
X_new_scaled = scaler.transform(X_new_final)
print(f"\n✓ Preprocessed {len(X_new_scaled)} rows for prediction")




✓ Preprocessed 71205 rows for prediction


In [17]:
print("\nMaking predictions...")
predictions = champion_model.predict(X_new_scaled)

# Add predictions as new column
df_new["sales"] = predictions


Making predictions...


In [18]:
output_csv = "group4.csv"  
df_new.to_csv(output_csv, index=False)
print(f"✓ Saved predictions to {output_csv}")

print(f"\nPrediction Statistics:")
print(f"  Mean:  ${predictions.mean():,.2f}")
print(f"  Min:   ${predictions.min():,.2f}")
print(f"  Max:   ${predictions.max():,.2f}")
print(f"  Std:   ${predictions.std():,.2f}")


✓ Saved predictions to group4.csv

Prediction Statistics:
  Mean:  $5,827.17
  Min:   $438.96
  Max:   $33,549.07
  Std:   $3,612.20


### save to CSV

In [19]:

output_csv = "group4.csv"  
df_new.to_csv(output_csv, index=False)
print(f"\n✓ Saved predictions to {output_csv}")
print(f"  Total rows: {len(df_new):,}")


✓ Saved predictions to group4.csv
  Total rows: 71,205


## Create TXT file with R² score from your test set

In [20]:

r2_test = 0.8915
output_txt = "group4.txt"
with open(output_txt, "w") as f:
    f.write(str(r2_test))
print(f"✓ Saved R² score to {output_txt}")

print("\n" + "="*50)
print("✅ PROCESS COMPLETED SUCCESSFULLY!")
print("="*50)

✓ Saved R² score to group4.txt

✅ PROCESS COMPLETED SUCCESSFULLY!
